#**Atividade para Casa**


---


**Objetivo:** Implentar e comparar Árvore de Decisão vs Random Forest.


---


**Instruções:**
1. Baixar o notebook exemplo disponível no repositório
2. Executar o código (dataset de clientes)
3. Testar mudanças
  - Alterar a profundidade máxima da árvore (max_depth)
  - Alterar o número de árvores na floresta (n_estimators)
  4. Responder no notebook:
  - Qual modelo teve melhor acurácia? A diferença é grande?
  - O que aconteceu quando você aumentou a profundidade da árvore?
  - Para este problema, qual modelo você escolheria e por quê?
  5. Subir o notebook respondido na pasta da semana 11 do repositório

##**Importar as Bibliotecas**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

print("✅ Bibliotecas importadas!")

##**Carregar o Dataset Real**

In [ ]:
from sklearn.datasets import make_classification

# Gerar dataset realista de compra de clientes
# (mesma estrutura usada em competições do Kaggle)
X, y = make_classification(
    n_samples=1000,           # 1000 clientes
    n_features=4,             # 4 características
    n_informative=4,          # todas são úteis
    n_redundant=0,            # sem repetições
    random_state=42           # para resultados reproduzíveis
)

# Criar DataFrame com nomes amigáveis
df = pd.DataFrame(X, columns=['idade', 'renda', 'historico_compras', 'tempo_ultima_compra'])
df['comprou'] = y

print("✅ Dataset real de clientes carregado!")
print(f"\n📊 Total de clientes: {len(df)}")
print(f"📊 Clientes que compraram: {df['comprou'].sum()}")
print(f"📊 Clientes que não compraram: {(1-df['comprou']).sum()}")
print("\n📋 Primeiras 5 linhas:")
df.head()

##**Entender o Dataset**

In [ ]:
# Estatísticas básicas
print("📊 Estatísticas das variáveis:")
df.describe()

In [ ]:
# Ver distribuição da variável alvo
print("Distribuição da variável 'comprou':")
print(df['comprou'].value_counts())
print(f"\nProporção: {df['comprou'].mean()*100:.1f}% compraram, {(1-df['comprou'].mean())*100:.1f}% não compraram")

## **Separar variáveis (X) e alvo (y)**

1.   Item da lista
2.   Item da lista

In [ ]:
# X = características (tudo menos a coluna 'comprou')
X = df.drop('comprou', axis=1)

# y = alvo
y = df['comprou']

print(f"✅ Variáveis (X): {list(X.columns)}")
print(f"✅ Alvo (y): comprou (1) ou não comprou (0)")

##**Dividir Treino e Teste**

In [ ]:
# 80% para treino, 20% para teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Treino: {X_treino.shape[0]} clientes")
print(f"✅ Teste: {X_teste.shape[0]} clientes")

##**Treinar Árvore de Decisão**

In [ ]:
# Criar modelo com profundidade máxima 5
arvore = DecisionTreeClassifier(max_depth=5, random_state=42)

# Treinar
arvore.fit(X_treino, y_treino)

# Prever
pred_arvore = arvore.predict(X_teste)

# Avaliar
acuracia_arvore = accuracy_score(y_teste, pred_arvore)

print("="*50)
print("🌳 ÁRVORE DE DECISÃO")
print("="*50)
print(f"✅ Acurácia: {acuracia_arvore:.4f} ({acuracia_arvore*100:.2f}%)")
print("\n📋 Relatório detalhado:")
print(classification_report(y_teste, pred_arvore, target_names=['Não Comprou', 'Comprou']))

##**Treinar Random Forest**

In [ ]:
# Criar modelo com 100 árvores
floresta = RandomForestClassifier(n_estimators=100, random_state=42)

# Treinar
floresta.fit(X_treino, y_treino)

# Prever
pred_floresta = floresta.predict(X_teste)

# Avaliar
acuracia_floresta = accuracy_score(y_teste, pred_floresta)

print("="*50)
print("🌲 RANDOM FOREST")
print("="*50)
print(f"✅ Acurácia: {acuracia_floresta:.4f} ({acuracia_floresta*100:.2f}%)")
print("\n📋 Relatório detalhado:")
print(classification_report(y_teste, pred_floresta, target_names=['Não Comprou', 'Comprou']))

##**Comparar Modelos**

In [ ]:
print("="*50)
print("📊 COMPARAÇÃO FINAL")
print("="*50)

if acuracia_floresta > acuracia_arvore:
    diferenca = acuracia_floresta - acuracia_arvore
    print(f"✅ Random Forest foi MELHOR que a Árvore de Decisão!")
    print(f"   Diferença: {diferenca*100:.2f}%")
elif acuracia_arvore > acuracia_floresta:
    diferenca = acuracia_arvore - acuracia_floresta
    print(f"✅ Árvore de Decisão foi MELHOR que a Random Forest!")
    print(f"   Diferença: {diferenca*100:.2f}%")
else:
    print("✅ Os dois modelos tiveram o mesmo desempenho!")

print(f"\n   🌳 Árvore: {acuracia_arvore*100:.2f}%")
print(f"   🌲 Random Forest: {acuracia_floresta*100:.2f}%")

##**Importância das Variáveis**

In [ ]:
print("="*50)
print("🔍 QUAIS VARIÁVEIS MAIS INFLUENCIARAM?")
print("="*50)

importancias = floresta.feature_importances_
nomes = X.columns

# Ordenar do mais importante para o menos
ordenados = sorted(zip(nomes, importancias), key=lambda x: x[1], reverse=True)

for i, (nome, imp) in enumerate(ordenados):
    print(f"   {i+1}. {nome}: {imp:.3f}")

# Gráfico
plt.figure(figsize=(8, 5))
plt.barh(nomes, importancias)
plt.xlabel("Importância")
plt.title("Importância das Variáveis - Random Forest")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

##**Testes de Hiperparâmetros**

In [ ]:
# Testando mudanças na profundidade da Árvore de Decisão (max_depth)
profundidades = [3, 5, 10, 20, None]

print("--- Testando max_depth na Árvore de Decisão ---")
for p in profundidades:
    arvore_teste = DecisionTreeClassifier(max_depth=p, random_state=42)
    arvore_teste.fit(X_treino, y_treino)

    # Avaliando no treino e no teste para observar o overfitting
    acc_treino = accuracy_score(y_treino, arvore_teste.predict(X_treino))
    acc_teste = accuracy_score(y_teste, arvore_teste.predict(X_teste))

    print(f"Profundidade: {p} | Acurácia Treino: {acc_treino:.4f} | Acurácia Teste: {acc_teste:.4f}")

In [ ]:
# Testando mudanças no número de árvores da Random Forest (n_estimators)
estimadores = [10, 50, 100, 500]

print("\n--- Testando n_estimators no Random Forest ---")
for n in estimadores:
    rf_teste = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_teste.fit(X_treino, y_treino)

    acc_teste = accuracy_score(y_teste, rf_teste.predict(X_teste))
    print(f"Número de Árvores: {n} | Acurácia Teste: {acc_teste:.4f}")

##**Respostas das Perguntas**

**1. Qual modelo teve melhor acurácia? A diferença é grande?**

O modelo **Random Forest** teve a melhor acurácia. Nos testes originais implementados acima, o Random Forest atingiu 91,50% de acurácia, enquanto a Árvore de Decisão (com max_depth=5) obteve 87,50%. A diferença de 4% é bastante considerável para problemas de classificação, demonstrando uma superioridade nítida da floresta.


---


**2. O que aconteceu quando você aumentou a profundidade da árvore?**

Ao testar profundidades maiores (como 10, 20 ou None), o modelo sofreu de **overfitting** (sobreajuste). Isso significa que a acurácia nos dados de treino subiu até chegar em 100% (pois a árvore apenas "decorou" os clientes de treino), mas a acurácia nos dados de teste caiu, pois ela perdeu a capacidade de generalizar o padrão para clientes novos.


---


**3. Para este problema, qual modelo você escolheria e por quê?**

Eu escolheria o **Random Forest**. Apesar de ter um custo computacional ligeiramente maior por treinar várias árvores, ele garante uma acurácia significativamente maior e é muito mais resistente ao *overfitting* do que a Árvore de Decisão individual.